# US Revenue Growth Ranking

**소스 테이블** : `us_revenue_forecast_data`  
**증가율 계산** : 과거 4분기 actual 합산 vs 향후 4분기 forecast 합산  
**forecast_date**: ticker별 가장 최근 예측일 자동 선택

```
증가율(%) = (향후 4Q 합산 - 과거 4Q 합산) / |과거 4Q 합산| × 100
```

---
| 셀 | 단계 |
|----|------|
| 1  | 환경 설정 & 경로 자동 감지 |
| 2  | 파라미터 설정 |
| 3  | DB 연결 |
| 4  | 매출 증가율 순위 조회 함수 |
| 5  | 실행 & 결과 출력 |

## Cell 1 · 환경 설정 & 경로 자동 감지

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",         # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",  # 데스크탑
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


## Cell 2 · 파라미터 설정

**여기만 수정하세요**

| 파라미터 | 설명 | 기본값 |
|----------|------|--------|
| `MODEL` | 예측 모델 선택 | `"SARIMA"` |
| `TOP_N` | 상위 몇 위까지 출력 | `50` |
| `ITEM` | 재무 항목 | `"sale"` |
| `FORECAST_DATE` | 특정 날짜 지정 (`None` = 최신 자동) | `None` |

**사용 가능한 모델**: `"SARIMA"` · `"ETS"` · `"Prophet"` · `"LSTM"` · `"Theta"` · `"Ensemble"`

In [23]:
# ══════════════════════════════════════════════════════
#  ★ 여기만 수정하세요 ★
# ══════════════════════════════════════════════════════

MODEL         = "SARIMA"   # 예측 모델: SARIMA / ETS / Prophet / LSTM / Theta / Ensemble
TOP_N         = 100        # 상위 N개 출력
ITEM          = "sale"     # 재무 항목
FORECAST_DATE = None       # None → 각 ticker별 최신 forecast_date 자동 선택
                           # 특정 날짜 지정 예: "2026-03-25"

DEST_TABLE    = "us_revenue_forecast_data"  # 예측 결과 테이블

# ══════════════════════════════════════════════════════
print(f"[파라미터]")
print(f"  MODEL         = {MODEL}")
print(f"  TOP_N         = {TOP_N}")
print(f"  ITEM          = {ITEM}")
print(f"  FORECAST_DATE = {FORECAST_DATE if FORECAST_DATE else '각 ticker 최신 자동'}")
print(f"  DEST_TABLE    = {DEST_TABLE}")


[파라미터]
  MODEL         = SARIMA
  TOP_N         = 100
  ITEM          = sale
  FORECAST_DATE = 각 ticker 최신 자동
  DEST_TABLE    = us_revenue_forecast_data


## Cell 3 · DB 연결

In [24]:
import pymysql
import pandas as pd
from typing import Optional
from datetime import datetime
from IPython.display import display
from DATA.config import get_db_info, get_engine

db_info = get_db_info()
engine  = get_engine(db_info)

# pymysql 직접 커서용 연결 함수 (pd.read_sql 버그 우회)
def get_conn():
    return pymysql.connect(
        host        = db_info["host"],
        port        = db_info.get("port", 3307),
        user        = db_info["user"],
        password    = db_info["password"],
        db          = db_info.get("database", "investar"),
        charset     = "utf8mb4",
        autocommit  = False,
        cursorclass = pymysql.cursors.DictCursor,
    )

# 연결 테스트
conn = get_conn()
with conn.cursor() as cur:
    cur.execute("SELECT 1")
conn.close()
print(f"[OK] DB 연결 성공  host={db_info.get('host')}  db={db_info.get('database')}")


[OK] DB 연결 성공  host=192.168.0.230  db=investar


## Cell 4 · 매출 증가율 순위 조회 함수

### 로직
```
1. 각 ticker의 가장 최근 forecast_date 확인
2. actual 마지막 4분기 합산  → past_4q
3. 해당 model 예측 첫 4분기 합산 → future_4q
4. 증가율 = (future_4q - past_4q) / |past_4q| × 100
5. 증가율 내림차순 정렬 → 상위 N개 반환
```

In [25]:
def get_revenue_growth_ranking(
    table_name: str        = DEST_TABLE,
    item: str              = "sale",
    model: str             = "SARIMA",
    top_n: int             = 10,
    forecast_date: Optional[str] = None,
) -> pd.DataFrame:
    """
    매출 증가율 상위 N개 ticker 를 반환합니다.

    테이블 스키마: ticker, item, date, data_type, model, value, forecast_date
      - data_type = 'actual'   → 실제값  (model 컬럼도 'actual')
      - data_type = 'forecast' → 예측값  (model = 'SARIMA' 등)

    증가율 = (향후 4Q 합산 - 과거 4Q 합산) / |과거 4Q 합산| × 100
    """
    # ── STEP 1: actual 마지막 4분기 조회 ────────────────────────
    # 각 ticker의 최신 forecast_date 기준 actual 행 전체를 가져온 뒤
    # Python 에서 마지막 4분기를 추출 (날짜 정렬 오류 방지)
    if forecast_date:
        fd_filter_actual   = "AND a.forecast_date = %s"
        fd_filter_forecast = "AND f.forecast_date = %s"
        fd_val = forecast_date
    else:
        fd_filter_actual   = ""
        fd_filter_forecast = ""
        fd_val = None

    # actual: 각 ticker의 최신 forecast_date 에 해당하는 실제값 전체
    sql_actual = f"""
        SELECT a.ticker, a.date, a.value, a.forecast_date
        FROM   {table_name} a
        INNER JOIN (
            SELECT ticker, MAX(forecast_date) AS max_fd
            FROM   {table_name}
            WHERE  item      = %s
              AND  data_type = 'actual'
              {fd_filter_actual}
            GROUP  BY ticker
        ) latest
          ON  a.ticker        = latest.ticker
          AND a.forecast_date = latest.max_fd
        WHERE a.item      = %s
          AND a.data_type = 'actual'
        ORDER BY a.ticker, a.date DESC
    """

    # forecast: 각 ticker의 최신 forecast_date 에 해당하는 예측값 전체
    sql_forecast = f"""
        SELECT f.ticker, f.date, f.value, f.forecast_date
        FROM   {table_name} f
        INNER JOIN (
            SELECT ticker, MAX(forecast_date) AS max_fd
            FROM   {table_name}
            WHERE  item      = %s
              AND  data_type = 'forecast'
              AND  model     = %s
              {fd_filter_forecast}
            GROUP  BY ticker
        ) latest
          ON  f.ticker        = latest.ticker
          AND f.forecast_date = latest.max_fd
        WHERE f.item      = %s
          AND f.data_type = 'forecast'
          AND f.model     = %s
        ORDER BY f.ticker, f.date ASC
    """

    # 파라미터 구성 (%s 순서에 맞게)
    if fd_val:
        params_actual   = (item, fd_val, item)
        params_forecast = (item, model, fd_val, item, model)
    else:
        params_actual   = (item, item)
        params_forecast = (item, model, item, model)

    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql_actual, params_actual)
            actual_rows = cur.fetchall()
            cur.execute(sql_forecast, params_forecast)
            forecast_rows = cur.fetchall()
    finally:
        conn.close()

    if not actual_rows:
        print(f"[WARN] actual 데이터 없음 (item={item})")
        return pd.DataFrame()
    if not forecast_rows:
        print(f"[WARN] forecast 데이터 없음 (item={item}, model={model})")
        return pd.DataFrame()

    df_actual   = pd.DataFrame(actual_rows)
    df_forecast = pd.DataFrame(forecast_rows)

    df_actual["date"]    = pd.to_datetime(df_actual["date"],    errors="coerce")
    df_actual["value"]   = pd.to_numeric(df_actual["value"],    errors="coerce")
    df_forecast["date"]  = pd.to_datetime(df_forecast["date"],  errors="coerce")
    df_forecast["value"] = pd.to_numeric(df_forecast["value"],  errors="coerce")

    # ── STEP 2: ticker별 과거 4Q 합산 (actual 마지막 4분기) ─────
    def _past_4q(grp):
        g = grp.sort_values("date", ascending=False)
        last4 = g.head(4)
        return pd.Series({
            "past_4q"      : last4["value"].sum(),
            "past_from"    : last4["date"].min(),
            "past_to"      : last4["date"].max(),
            "forecast_date": g["forecast_date"].iloc[0],
            "actual_count" : len(g),    # 전체 actual 분기 수 (참고용)
        })

    past_df = (
        df_actual
        .groupby("ticker", group_keys=False)
        .apply(_past_4q)
        .reset_index()
    )

    # ── STEP 3: ticker별 향후 4Q 합산 (forecast 첫 4분기) ───────
    def _future_4q(grp):
        g = grp.sort_values("date", ascending=True)
        first4 = g.head(4)
        return pd.Series({
            "future_4q"  : first4["value"].sum(),
            "future_from": first4["date"].min(),
            "future_to"  : first4["date"].max(),
        })

    future_df = (
        df_forecast
        .groupby("ticker", group_keys=False)
        .apply(_future_4q)
        .reset_index()
    )

    # ── STEP 4: 합산 후 증가율 계산 ─────────────────────────────
    merged = pd.merge(past_df, future_df, on="ticker", how="inner")

    # 과거 합산 0 이하 또는 NaN ticker 제외
    merged = merged[
        merged["past_4q"].notna()  &
        merged["future_4q"].notna() &
        (merged["past_4q"].abs() > 0)
    ].copy()

    merged["growth_pct"] = (
        (merged["future_4q"] - merged["past_4q"]) /
        merged["past_4q"].abs() * 100
    ).round(2)

    # 기간 문자열 생성
    merged["past_period"]   = merged["past_from"].dt.date.astype(str) + " ~ " + merged["past_to"].dt.date.astype(str)
    merged["future_period"] = merged["future_from"].dt.date.astype(str) + " ~ " + merged["future_to"].dt.date.astype(str)

    # ── STEP 5: 정렬 & 상위 N개 ──────────────────────────────────
    result = (
        merged
        .sort_values("growth_pct", ascending=False)
        .head(top_n)
        [[
            "ticker", "growth_pct",
            "past_4q", "future_4q",
            "past_period", "future_period",
            "forecast_date", "actual_count",
        ]]
        .reset_index(drop=True)
    )
    result.index = result.index + 1
    result.index.name = "rank"
    return result


print("[OK] get_revenue_growth_ranking 함수 정의 완료")
print(f"     스키마: ticker / item / date / data_type / model / value / forecast_date")


[OK] get_revenue_growth_ranking 함수 정의 완료
     스키마: ticker / item / date / data_type / model / value / forecast_date


## Cell 5 · 실행 & 결과 출력

In [26]:
print(f"[실행] 매출 증가율 TOP {TOP_N} 조회")
print(f"       모델={MODEL}  항목={ITEM}  forecast_date={FORECAST_DATE or '최신 자동'}")
print("-" * 60)

ranking_df = get_revenue_growth_ranking(
    table_name    = DEST_TABLE,
    item          = ITEM,
    model         = MODEL,
    top_n         = TOP_N,
    forecast_date = FORECAST_DATE,
)

if ranking_df.empty:
    print("[결과 없음] 파라미터(MODEL, ITEM, FORECAST_DATE)를 확인하세요.")
else:
    # ── 표시용 포맷 ──────────────────────────────────────────
    disp = ranking_df.copy()
    disp["past_4q"]    = disp["past_4q"].apply(lambda x: f"{x:>20,.0f}")
    disp["future_4q"]  = disp["future_4q"].apply(lambda x: f"{x:>20,.0f}")
    disp["growth_pct"] = disp["growth_pct"].apply(lambda x: f"{x:+.2f}%")
    disp.columns = [
        "ticker", "증가율", "과거4Q합산", "향후4Q합산",
        "과거기간(4Q)", "예측기간(4Q)", "forecast_date", "보유분기수"
    ]

    print(f"\n[결과] 매출 증가율 TOP {len(ranking_df)}  (모델: {MODEL} / 항목: {ITEM})")
    display(disp)

    # ── 요약 통계 ─────────────────────────────────────────────
    g = ranking_df["growth_pct"]
    top1 = ranking_df.iloc[0]
    print(f"\n[통계]")
    print(f"  1위  : {top1['ticker']:8s}  증가율 {top1['growth_pct']:+.2f}%")
    print(f"  평균 : {g.mean():+.2f}%")
    print(f"  중위 : {g.median():+.2f}%")
    print(f"  최소 : {g.min():+.2f}%  ({ranking_df.loc[g.idxmin(), 'ticker']})")


[실행] 매출 증가율 TOP 100 조회
       모델=SARIMA  항목=sale  forecast_date=최신 자동
------------------------------------------------------------

[결과] 매출 증가율 TOP 100  (모델: SARIMA / 항목: sale)


,ticker,증가율,과거4Q합산,향후4Q합산,과거기간(4Q),예측기간(4Q),forecast_date,보유분기수
rank,,,,,,,,
1,CMRX,+7042.34%,"212,000","15,141,751",2024-03-31 ~ 2024-12-31,2025-03-31 ~ 2025-12-31,2026-04-04,40
2,NETE,+2625.37%,"-7,870,625","198,762,286",2021-09-30 ~ 2025-06-30,2025-09-30 ~ 2026-06-30,2026-03-30,30
3,GLYC,+1758.38%,"54,086","1,005,124",2024-12-31 ~ 2025-09-30,2025-12-31 ~ 2026-09-30,2026-03-31,43
4,DBVT,+930.03%,"511,000","5,263,462",2024-12-31 ~ 2025-09-30,2025-12-31 ~ 2026-09-30,2026-03-31,43
5,DISH,+793.38%,"3,290,519,485,000","29,396,749,391,787",2024-12-31 ~ 2025-09-30,2025-12-31 ~ 2026-09-30,2026-04-03,40
...,...,...,...,...,...,...,...,...
96,CNX,+42.49%,"2,258,570,000","3,218,251,718",2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-03-30,44
97,AGIO,+42.03%,"54,028,000","76,737,928",2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-03,40
98,RDY,+41.50%,"345,831,000,000","489,354,042,960",2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-03,40



[통계]
  1위  : CMRX      증가율 +7042.34%
  평균 : +225.38%
  중위 : +66.30%
  최소 : +39.95%  (SCZM)


In [31]:
ranking_df.loc[50:]

,ticker,growth_pct,past_4q,future_4q,past_period,future_period,forecast_date,actual_count
rank,,,,,,,,
50,URG,67.56,2.720700e+07,4.558857e+07,2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-04,40
51,TRX,65.04,9.780042e+07,1.614121e+08,2025-02-28 ~ 2025-11-30,2025-12-31 ~ 2026-09-30,2026-04-04,40
52,YPF,62.29,2.350496e+13,3.814726e+13,2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-03,40
53,ALNY,61.38,3.713937e+09,5.993393e+09,2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-03,40
54,RGLD,60.67,1.028709e+09,1.652811e+09,2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-03,40
55,DNOW,60.10,2.820000e+09,4.514936e+09,2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-03,40
56,RWT,59.08,2.431480e+08,3.868042e+08,2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-03-31,44
57,BLFS,58.52,7.400500e+07,1.173096e+08,2024-12-31 ~ 2025-09-30,2025-12-31 ~ 2026-09-30,2026-03-31,43
58,MNKD,58.23,3.489660e+08,5.521523e+08,2025-03-31 ~ 2025-12-31,2026-03-31 ~ 2026-12-31,2026-04-03,40


## Cell 6 · 모델별 비교 (선택)

여러 모델의 Top N 을 나란히 비교합니다.

In [27]:
# ── 비교할 모델 목록 ──────────────────────────────────────────
COMPARE_MODELS = ["SARIMA", "ETS", "Theta", "Ensemble"]
COMPARE_TOP_N  = 10

print(f"[모델별 TOP {COMPARE_TOP_N} 비교]  항목={ITEM}")
print("=" * 60)

for m in COMPARE_MODELS:
    df_m = get_revenue_growth_ranking(
        table_name    = DEST_TABLE,
        item          = ITEM,
        model         = m,
        top_n         = COMPARE_TOP_N,
        forecast_date = FORECAST_DATE,
    )
    if df_m.empty:
        print(f"\n[{m}] 데이터 없음")
        continue

    compact = df_m[["ticker", "growth_pct"]].copy()
    compact["growth_pct"] = compact["growth_pct"].apply(lambda x: f"{x:+.2f}%")
    compact.columns = ["ticker", "증가율"]
    print(f"\n[{m}] TOP {COMPARE_TOP_N}")
    display(compact)


[모델별 TOP 10 비교]  항목=sale

[SARIMA] TOP 10


,ticker,증가율
rank,,
1,CMRX,+7042.34%
2,NETE,+2625.37%
3,GLYC,+1758.38%
4,DBVT,+930.03%
5,DISH,+793.38%
6,AD,+575.03%
7,BLU,+544.35%
8,USM,+449.20%
9,TGTX,+445.11%



[ETS] TOP 10


,ticker,증가율
rank,,
1,CMRX,+2732.93%
2,NR,+1321.81%
3,DISH,+893.23%
4,KALV,+800.51%
5,PTSI,+740.95%
6,RCAT,+561.71%
7,SAGE,+452.45%
8,QIWI,+438.57%
9,UNIT,+332.52%



[Theta] TOP 10


,ticker,증가율
rank,,
1,CMRX,+2385.95%
2,NR,+833.63%
3,PTSI,+691.30%
4,DBVT,+420.35%
5,QIWI,+335.44%
6,VVI,+282.23%
7,PRSU,+243.27%
8,KALV,+220.80%
9,TGTX,+200.11%



[Ensemble] TOP 10


,ticker,증가율
rank,,
1,CMRX,+4053.74%
2,NR,+737.01%
3,NETE,+726.83%
4,DBVT,+547.87%
5,DISH,+529.13%
6,KALV,+510.66%
7,QIWI,+387.00%
8,RCAT,+299.67%
9,TGTX,+289.34%


In [44]:
# ══════════════════════════════════════════════════════
#  검증 파라미터 — 여기만 수정하세요
# ══════════════════════════════════════════════════════
VERIFY_TICKER        = "LLY"    # ← 확인할 티커
VERIFY_MODEL         = MODEL     # Cell 2 의 MODEL 사용 (직접 변경 가능)
VERIFY_ITEM          = ITEM      # Cell 2 의 ITEM 사용
VERIFY_FORECAST_DATE = None      # None → 최신 forecast_date 자동, 또는 "2026-03-25"
# ══════════════════════════════════════════════════════


def get_ticker_timeseries(
    ticker: str,
    item: str             = "sale",
    model: str            = "SARIMA",
    forecast_date         = None,
    table_name: str       = DEST_TABLE,
) -> pd.DataFrame:
    """
    단일 티커의 actual + forecast 전체 시계열을 반환합니다.

    Returns
    -------
    DataFrame  columns: date, data_type, model, value, used_in_calc, forecast_date
      used_in_calc : '과거4Q★' / '향후4Q★' / '' — 증가율 계산에 사용된 4분기 표시
    """
    # ── 최신 forecast_date 확인 ──────────────────────────────
    if forecast_date:
        fd = forecast_date
    else:
        conn = get_conn()
        try:
            with conn.cursor() as cur:
                cur.execute(
                    f"SELECT MAX(forecast_date) AS max_fd FROM {table_name}"
                    f" WHERE ticker=%s AND item=%s",
                    (ticker, item)
                )
                row = cur.fetchone()
                fd = str(row['max_fd']) if row and row['max_fd'] else None
        finally:
            conn.close()

    if not fd:
        print(f"[WARN] {ticker}: forecast_date 없음")
        return pd.DataFrame()

    # ── actual + forecast 조회 ───────────────────────────────
    sql = f"""
        SELECT date, data_type, model, value, forecast_date
        FROM   {table_name}
        WHERE  ticker        = %s
          AND  item          = %s
          AND  forecast_date = %s
          AND  (
                (data_type = 'actual')
            OR  (data_type = 'forecast' AND model = %s)
          )
        ORDER  BY data_type DESC, date ASC
    """
    # data_type DESC → actual 이 forecast 보다 먼저 (a > f 알파벳)

    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker, item, fd, model))
            rows = cur.fetchall()
    finally:
        conn.close()

    if not rows:
        print(f"[WARN] {ticker}: 데이터 없음 (item={item}, model={model}, forecast_date={fd})")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df['date']  = pd.to_datetime(df['date'],  errors='coerce')
    df['value'] = pd.to_numeric(df['value'],  errors='coerce')
    df = df.dropna(subset=['date']).sort_values(['data_type', 'date'], ascending=[False, True])

    # ── 증가율 계산에 사용된 4Q 범위 표시 ───────────────────────
    df['used_in_calc'] = ''

    # 과거 4Q: actual 행 중 날짜 내림차순 상위 4개
    actual_idx = df[df['data_type'] == 'actual'].sort_values('date', ascending=False).head(4).index
    df.loc[actual_idx, 'used_in_calc'] = '과거4Q★'

    # 향후 4Q: forecast 행 중 날짜 오름차순 상위 4개
    fc_idx = df[df['data_type'] == 'forecast'].sort_values('date', ascending=True).head(4).index
    df.loc[fc_idx, 'used_in_calc'] = '향후4Q★'

    # ── 증가율 직접 계산 (검증용) ───────────────────────────────
    past_sum   = df.loc[actual_idx, 'value'].sum()
    future_sum = df.loc[fc_idx,     'value'].sum()
    growth     = (future_sum - past_sum) / abs(past_sum) * 100 if past_sum != 0 else float('nan')

    df = df[['date', 'data_type', 'model', 'value', 'used_in_calc', 'forecast_date']].reset_index(drop=True)
    df.index = df.index + 1

    return df, past_sum, future_sum, growth, fd


# ── 실행 & 출력 ──────────────────────────────────────────────
result = get_ticker_timeseries(
    ticker        = VERIFY_TICKER,
    item          = VERIFY_ITEM,
    model         = VERIFY_MODEL,
    forecast_date = VERIFY_FORECAST_DATE,
)

if isinstance(result, tuple):
    ts_df, past_sum, future_sum, growth, fd = result

    print(f"[검증] {VERIFY_TICKER}  |  모델={VERIFY_MODEL}  |  항목={VERIFY_ITEM}")
    print(f"       forecast_date = {fd}")
    print(f"       전체 행 수    = {len(ts_df)}행  "
          f"(actual={len(ts_df[ts_df['data_type']=='actual'])}  "
          f"forecast={len(ts_df[ts_df['data_type']=='forecast'])})")
    print("-" * 70)

    # 포맷 적용
    disp = ts_df.copy()
    disp['value'] = disp['value'].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else '')
    disp.columns = ['날짜', '구분', '모델', '매출값', '계산사용여부', 'forecast_date']
    display(disp)

    # 증가율 검증 출력
    print(f"\n[증가율 검증]")
    print(f"  과거 4Q 합산  : {past_sum:>20,.0f}")
    print(f"  향후 4Q 합산  : {future_sum:>20,.0f}")
    print(f"  증가율        : {growth:+.2f}%")
    print(f"\n  → 랭킹 조회 결과와 일치하면 계산이 정확합니다.")

[검증] LLY  |  모델=SARIMA  |  항목=sale
       forecast_date = 2026-04-03
       전체 행 수    = 48행  (actual=40  forecast=8)
----------------------------------------------------------------------


,날짜,구분,모델,매출값,계산사용여부,forecast_date
1,2026-03-31,forecast,SARIMA,"18,934,166,196",향후4Q★,2026-04-03
2,2026-06-30,forecast,SARIMA,"21,847,547,729",향후4Q★,2026-04-03
3,2026-09-30,forecast,SARIMA,"24,048,169,006",향후4Q★,2026-04-03
4,2026-12-31,forecast,SARIMA,"25,965,289,215",향후4Q★,2026-04-03
5,2027-03-31,forecast,SARIMA,"26,208,440,106",,2026-04-03
6,2027-06-30,forecast,SARIMA,"29,201,808,920",,2026-04-03
7,2027-09-30,forecast,SARIMA,"31,615,458,571",,2026-04-03
8,2027-12-31,forecast,SARIMA,"33,829,946,332",,2026-04-03
9,2016-03-31,actual,actual,"4,865,100,000",,2026-04-03
10,2016-06-30,actual,actual,"5,404,800,000",,2026-04-03



[증가율 검증]
  과거 4Q 합산  :       65,179,000,000
  향후 4Q 합산  :       90,795,172,147
  증가율        : +39.30%

  → 랭킹 조회 결과와 일치하면 계산이 정확합니다.
